In [1]:
import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

import json

d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
from utils.navigation import Agent, Maze, explore_maze
from utils.actions import resume_notes
from utils.utils import get_survey, get_advices

# Settings

In [ ]:
# test = "gemini-3-flash-preview"
test = "ministral-3-3b"
iterations = 25


output_dir = os.path.join("outputs", test)
os.makedirs(output_dir, exist_ok=True)

# Run Exploration

In [5]:
# Remote Gemini
# model = ChatGoogleGenerativeAI(model=test, request_timeout=180, max_retries=3)

# Run locally on LMStudio
model = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1", 
    api_key="not-needed", 
    model=test,
    request_timeout=180, max_retries=5,
    max_tokens=8192, 
    seed=42,
    model_kwargs={"response_format": {"type": "json_object"}}
)

# Prepare data structure and load existing data if available
params = ["travel_logs", "decision_logs", "analysis_logs", "last_notes", "end_causes"]

data = {}
for key in params:
    file_path = os.path.join(output_dir, f"{key}.json")
    
    if os.path.exists(file_path):
        with open(file_path, 'r') as f:
            data[key] = json.load(f)
    else:
        data[key] = []

In [ ]:
initial_iteration = len(data["travel_logs"]) 
last_iteration = initial_iteration + iterations


for iter in range(initial_iteration, last_iteration):

    print( f"\n--- Iteration {iter} ---\n")
        
    maze = Maze()
    agent = Agent(model=model)

    # Inject past notes
    if len(data["last_notes"]) > 0:

        # # 1 Cumulative notes from all past attempts
        # notes = "\n".join(f"Attempt {note['iteration']}: {note['data']['advice_for_future_self']}" for note in data["last_notes"])
        # strategy = resume_notes(agent, notes)

        # # 2 Iterating only last note
        # last_note = data["last_notes"][-1]['data']
        # strategy = last_note['advice_for_future_self']
        # print(f"Extracted Strategy from Past Notes:\n{strategy}\n")

        # # 3 Iterating only last survey
        # last_note = data["last_notes"][-1]['data']
        # strategy = get_survey(last_note)

        # 4 Get last n notes
        agent.last_notes = get_advices(data, 3)

    # Run Iteration
    new_data, agent = explore_maze(agent, maze, max_steps=32)

    # Store outputs for next run
    for key, new_items in new_data.items():
        if key in data:
            data[key].append({"iteration": iter, "data": new_items})
        else:
            data[key] = [{"iteration": iter, "data": new_items}]

    # Save data to file
    for key, content in data.items():
        filename = os.path.join(output_dir, f"{key}.json")
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(content, f, indent=4)



--- Iteration 1 ---



d:\nerii\Desktop\Work\DVS\Manson Maze\py\env\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PrologueAnalysis(meta_obs...:', ready_to_start=True), input_type=PrologueAnalysis])
  return self.__pydantic_serializer__.to_python(


AI Ready: True
